## Tools
### Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
### 1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
### 2. A function or coroutine to execute.

In [3]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

model = init_chat_model("gpt-4.1")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots "talk" because they are highly intelligent birds with the ability to mimic sounds they hear in their environment, including human speech. Here’s why:\n\n### **1. Social Nature**\nParrots are very social animals. In the wild, they live in flocks and rely on vocalizations to communicate with each other—for warning, bonding, mating, and coordinating activities. When kept as pets or in captivity, they may see humans as part of their "flock" and try to communicate the same way.\n\n### **2. Mimicry Ability**\nParrots have a unique vocal organ called the **syrinx**, which allows them to produce a wide range of sounds. Their brains are also wired for imitation and learning, making them excellent vocal mimics compared to most other animals.\n\n### **3. Attention and Interaction**\nParrots quickly learn that making certain sounds (such as words or phrases humans frequently say) gets them attention, treats, or other forms of interaction. This reinforces their talking be

In [4]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """ Get the weather at a location"""
    return f"It is sunny in {location}"

model_with_tools = model.bind_tools([get_weather])


In [7]:
response = model_with_tools.invoke("What's the weather like in Boston")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool name : {tool_call['name']}")
    print(f"Tool name : {tool_call['args']}")



content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 50, 'total_tokens': 64, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_62f6f64af5', 'id': 'chatcmpl-DxxXh1ZIzVCV6Wnur2mvqoAU0k63Q', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019f2df7-3a58-7200-b9a6-c63b71837517-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_79CGShPaOYkdkE9Nl00xsXB4', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 50, 'output_tokens': 14, 'total_tokens': 64, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Tool name : get_weather
Tool na

In [8]:
#step 1: Model generates tool calls
messages = [{"role":"user", "content":"What's the weather in Boston"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#step 2: Execute tools and collect results

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#step 3: Pass results back to model for final response

final_response = model_with_tools.invoke(messages)
print(final_response.text)


The weather in Boston is sunny right now. If you need more details like temperature or forecast, let me know!


In [9]:
messages

[{'role': 'user', 'content': "What's the weather in Boston"},
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 49, 'total_tokens': 63, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_62f6f64af5', 'id': 'chatcmpl-DxxdKQsWwTh49uzwdS8aliDVNz41C', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f2dfc-8f1b-71f3-9561-83b7e7f124f4-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_B3iUf1LFMY8XOV5pgPwPmlu9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 14, 'total_tokens': 63, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'outpu